In [23]:
import tapas.datasets
import tapas.generators
import tapas.threat_models
import tapas.attacks
import tapas.report

In [28]:
#load data
#dataset must be stored in the data folder as .csv with a .json file describing variable type
data = tapas.datasets.TabularDataset.read("data/dummy")

In [29]:
#see full dataset
print(data.data)
#see subset of data set
print(data.get_records([0,1]).data)

    age eye_color
0   4.0      blue
1   5.0     green
2   5.0      blue
3   2.0     green
4   3.0     brown
5   7.0      blue
6  77.0     brown
   age eye_color
0  4.0      blue
1  5.0     green


In [30]:
#split data into public and private datsets
attacker_data, defender_data = data.create_subsets(n = 2, sample_size= int(len(data) / 2))
print(defender_data.data)
print(attacker_data.data)

    age eye_color
6  77.0     brown
1   5.0     green
4   3.0     brown
   age eye_color
3  2.0     green
0  4.0      blue
2  5.0      blue


In [31]:
#create synthetic data
generator = tapas.generators.Raw()
generator.fit(defender_data)
synthetic_data = generator.generate(num_samples= len(defender_data))
synthetic_data.data

,age,eye_color
4,3.0,brown
6,77.0,brown
1,5.0,green


In [32]:
#define attacker knowledge
data_knowledge = tapas.threat_models.AuxiliaryDataKnowledge(
      test_data= defender_data,
      aux_data= attacker_data,
      num_training_records= len(attacker_data)
)

sdg_knowledge = tapas.threat_models.BlackBoxKnowledge(
    generator,
    num_synthetic_records= len(defender_data),
)

In [33]:
threat_model = tapas.threat_models.TargetedMIA(
   attacker_knowledge_data=data_knowledge,
   target_record=defender_data.get_records([0]),
   attacker_knowledge_generator=sdg_knowledge,
   generate_pairs=True,
   replace_target=True
)

c:\Users\arthe\repo\Benchmarking-of-Tabular-Synthetic-Data-Generation\.conda\lib\site-packages\tapas\threat_models\mia.py:206: UserWarning: 1 target record(s) were found in the auxiliary data. This is not recommended: it is best to remove target records to avoid duplicates and ensure that the task of membership inference is meaningful.
  warnings.warn(


In [34]:
from sklearn.ensemble import RandomForestClassifier
attacker = tapas.attacks.GroundhogAttack()

In [35]:
defender_data.as_numeric

array([[77.,  0.,  0.,  1.],
       [ 5.,  1.,  0.,  0.],
       [ 3.,  0.,  0.,  1.]])

In [38]:
#generate shadow datasets and train the classifier
attacker.train(threat_model, num_samples=2)

c:\Users\arthe\repo\Benchmarking-of-Tabular-Synthetic-Data-Generation\.conda\lib\site-packages\numpy\lib\function_base.py:2829: RuntimeWarning: invalid value encountered in true_divide
  c /= stddev[:, None]
c:\Users\arthe\repo\Benchmarking-of-Tabular-Synthetic-Data-Generation\.conda\lib\site-packages\numpy\lib\function_base.py:2830: RuntimeWarning: invalid value encountered in true_divide
  c /= stddev[None, :]


In [41]:
attack_summary = threat_model.test(attacker, num_samples = 100)
metrics = attack_summary.get_metrics()
print("Results:\n", metrics.head())

Results:
             dataset target_id generator     attack  accuracy  \
0  data/dummy (AUX)         0       Raw  Groundhog       0.5   

   true_positive_rate  false_positive_rate  mia_advantage  privacy_gain   auc  \
0                 1.0                  1.0            0.0           1.0  0.61   

   effective_epsilon  
0                inf  


c:\Users\arthe\repo\Benchmarking-of-Tabular-Synthetic-Data-Generation\.conda\lib\site-packages\numpy\lib\function_base.py:2829: RuntimeWarning: invalid value encountered in true_divide
  c /= stddev[:, None]
c:\Users\arthe\repo\Benchmarking-of-Tabular-Synthetic-Data-Generation\.conda\lib\site-packages\numpy\lib\function_base.py:2830: RuntimeWarning: invalid value encountered in true_divide
  c /= stddev[None, :]
c:\Users\arthe\repo\Benchmarking-of-Tabular-Synthetic-Data-Generation\.conda\lib\site-packages\numpy\lib\function_base.py:2829: RuntimeWarning: invalid value encountered in true_divide
  c /= stddev[:, None]
c:\Users\arthe\repo\Benchmarking-of-Tabular-Synthetic-Data-Generation\.conda\lib\site-packages\numpy\lib\function_base.py:2830: RuntimeWarning: invalid value encountered in true_divide
  c /= stddev[None, :]
